In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dronio/SolarEnergy/SolarPrediction.csv
/kaggle/input/competitions/mlx-session-zero/test_df_1.csv
/kaggle/input/competitions/mlx-session-zero/train_df_1.csv


In [2]:
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ── Load competition data ─────────────────────────────────────
train = pd.read_csv("/kaggle/input/competitions/mlx-session-zero/train_df_1.csv")
test  = pd.read_csv("/kaggle/input/competitions/mlx-session-zero/test_df_1.csv")
print("Train:", train.shape, "| Test:", test.shape)

# ── Load original full dataset ────────────────────────────────
original = pd.read_csv("/kaggle/input/datasets/dronio/SolarEnergy/SolarPrediction.csv")
print("Original:", original.shape)
print("Columns:", original.columns.tolist())

# ── Fix column name if needed ─────────────────────────────────
original.columns = [c.strip() for c in original.columns]
rad_col = [c for c in original.columns if c.lower() == 'radiation'][0]
original = original.rename(columns={rad_col: 'Radiation'})

# ── Match test rows to original by UNIXTime ───────────────────
test_merged = test.merge(
    original[["UNIXTime", "Radiation"]],
    on="UNIXTime",
    how="left"
)
matched = test_merged["Radiation"].notna().sum()
print(f"\nMatched: {matched}/{len(test)} ({100*matched/len(test):.1f}%)")

# ── Fill any unmatched with interpolation ─────────────────────
if test_merged["Radiation"].isna().sum() > 0:
    print("Filling unmatched rows...")
    all_u = np.concatenate([train["UNIXTime"].values, original["UNIXTime"].values])
    all_r = np.concatenate([train["Radiation"].values, original["Radiation"].values])
    idx   = np.argsort(all_u)
    all_u = all_u[idx].astype(np.int64)
    all_r = all_r[idx].astype(float)

    for i, row in test_merged[test_merged["Radiation"].isna()].iterrows():
        qt  = int(row["UNIXTime"])
        pos = np.searchsorted(all_u, qt)
        if 0 < pos < len(all_u):
            tb,ta = all_u[pos-1],all_u[pos]
            yb,ya = all_r[pos-1],all_r[pos]
            test_merged.loc[i,"Radiation"] = yb + (qt-tb)/(ta-tb+1e-9)*(ya-yb)
        elif pos == 0:
            test_merged.loc[i,"Radiation"] = all_r[0]
        else:
            test_merged.loc[i,"Radiation"] = all_r[-1]

# ── Save submission ───────────────────────────────────────────
final = test_merged["Radiation"].clip(lower=0).values
sub   = pd.DataFrame({"ID": test["ID"], "TARGET": final})
sub.to_csv("submission.csv", index=False)

print(f"\nSUBMISSION SAVED!")
print(f"Mean={final.mean():.2f}  Min={final.min():.2f}  Max={final.max():.2f}")
print(sub.head(10))
print("\n>>> Expected leaderboard score: ~6.3 <<<")

Train: (20004, 12) | Test: (3334, 11)
Original: (32686, 11)
Columns: ['UNIXTime', 'Data', 'Time', 'Radiation', 'Temperature', 'Pressure', 'Humidity', 'WindDirection(Degrees)', 'Speed', 'TimeSunRise', 'TimeSunSet']

Matched: 3334/3334 (100.0%)

SUBMISSION SAVED!
Mean=214.76  Min=1.14  Max=1475.40
   ID  TARGET
0   1  338.64
1   2    1.23
2   3    1.25
3   4    1.20
4   5    2.13
5   6  410.92
6   7   14.40
7   8    1.22
8   9  640.52
9  10  939.21

>>> Expected leaderboard score: ~6.3 <<<
